# Recursive Multi-Step Forecasting from One-Step Models

Este notebook toma como base el enfoque de `bigru_lstm_ari_univariate_experiment.ipynb` y agrega:

- entrenamiento **one-step** para `BiGRU`, `LSTM` y `ARIMA`
- inferencia **multi-step recursiva**
- ejecucion de **10 runs**
- almacenamiento de metricas en **Excel** por modelo, run, iteracion y variable
- hojas extra con metricas por horizonte

Estructura del Excel generado:

- `BiGRU_summary`
- `BiGRU_horizon`
- `LSTM_summary`
- `LSTM_horizon`
- `ARIMA_summary`
- `ARIMA_horizon`
- `configuration`


## 1. Dependencias

Si ejecutas este notebook en un entorno nuevo, instala primero las dependencias necesarias.


In [ ]:
# Ejecuta esta celda solo si tu entorno no tiene las dependencias instaladas.
# %pip install pmdarima openpyxl

## 2. Imports y configuracion

Los parametros principales estan centralizados aqui para poder ajustar el horizonte, numero de runs, variables objetivo y configuracion de modelos.


In [ ]:
from pathlib import Path
import random
import warnings
from time import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import tensorflow as tf
from pmdarima import ARIMA
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.metrics import MeanAbsoluteError, MeanAbsolutePercentageError
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import RMSprop

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-white")
pd.set_option("display.max_columns", 100)

DATA_FILE = Path("data.csv")
OUTPUT_DIR = Path("outputs_recursive_multistep")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_PATH = OUTPUT_DIR / "metrics_summary.xlsx"

RUNS = 10
NUM_FOLDS = 4
TRAIN_WITHIN_FOLD_RATIO = 0.8
INPUT_LENGTH = 1
OUTPUT_LENGTH = 1
FORECAST_HORIZON = 12
BASE_SEED = 42

TARGET_COLUMNS = None  # None -> usa todas las columnas
CUTOFF_DATE = "2023-07-05 00:00:00"

BIGRU_CONFIG = {
    "gru_units_1": 128,
    "gru_units_2": 64,
    "learning_rate": 5e-5,
    "epochs": 200,
    "batch_size": 256,
    "patience": 2,
}

LSTM_CONFIG = {
    "lstm_units_1": 128,
    "lstm_units_2": 128,
    "lstm_units_3": 64,
    "dense_units": 16,
    "dropout": 0.1,
    "learning_rate": 5e-5,
    "epochs": 250,
    "batch_size": 256,
    "patience": 2,
}

ARIMA_ORDER = (1, 1, 1)
ARIMA_SEASONAL_ORDER = (1, 1, 1, 7)

if OUTPUT_LENGTH != 1:
    raise ValueError("Para forecast recursivo basado en modelos one-step, OUTPUT_LENGTH debe ser 1.")

print(f"Archivo de datos: {DATA_FILE.resolve()}")
print(f"Directorio de salida: {OUTPUT_DIR.resolve()}")
print(f"Excel de metricas: {EXCEL_PATH.resolve()}")

## 3. Funciones auxiliares

Se definen funciones para:

- reproducibilidad
- generacion de folds temporales
- construccion de datasets supervisados
- escalado por variable
- metricas resumen y por horizonte
- construccion de modelos
- forecast recursivo


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def root_mean_squared_error(y_true, y_pred):
    return tf.math.sqrt(tf.math.reduce_mean(tf.square(y_pred - y_true)))


def create_supervised_dataset(series_values, input_length: int, output_length: int):
    x_data = []
    y_data = []
    rows = len(series_values)
    for idx in range(rows - input_length - output_length + 1):
        x_data.append(series_values[idx: idx + input_length])
        y_data.append(series_values[idx + input_length: idx + input_length + output_length])
    return np.array(x_data), np.array(y_data)


def build_temporal_folds(df: pd.DataFrame, num_folds: int, train_ratio: float):
    folds = {}
    cut_dates = {}
    len_data = len(df) / num_folds

    for iteration, _ in enumerate(range(0, num_folds - 1)):
        len_train_data = int(len_data * (iteration + 1))
        train_size = int(len_train_data * train_ratio)
        val_size = int(len_train_data * (1 - train_ratio))

        tr = df.iloc[:train_size].copy()
        vl = df.iloc[train_size: train_size + val_size].copy()
        ts = df.iloc[len_train_data: len_train_data + int(len_data)].copy()

        folds[iteration] = {
            "train": tr,
            "val": vl,
            "test": ts,
        }

        cut_dates[iteration] = {
            "train_end": tr.index[-1],
            "val_end": vl.index[-1],
            "test_end": ts.index[-1],
        }

    return folds, cut_dates


def prepare_univariate_fold_data(tr_series, vl_series, ts_series, input_length: int, output_length: int, forecast_horizon: int):
    scaler = MinMaxScaler(feature_range=(-1, 1))

    train_raw = tr_series.to_numpy(dtype=float).reshape(-1, 1)
    val_raw = vl_series.to_numpy(dtype=float).reshape(-1, 1)
    test_raw = ts_series.to_numpy(dtype=float).reshape(-1, 1)

    scaler.fit(train_raw)

    train_scaled = scaler.transform(train_raw).reshape(-1)
    val_scaled = scaler.transform(val_raw).reshape(-1)
    test_scaled = scaler.transform(test_raw).reshape(-1)

    x_tr, y_tr = create_supervised_dataset(train_scaled, input_length, output_length)
    x_vl, y_vl = create_supervised_dataset(val_scaled, input_length, output_length)
    x_ts, y_ts = create_supervised_dataset(test_scaled, input_length, output_length)

    history_before_test_scaled = np.concatenate([train_scaled, val_scaled])
    available_horizon = min(forecast_horizon, len(test_raw))

    return {
        "scaler": scaler,
        "train_raw": train_raw.reshape(-1),
        "val_raw": val_raw.reshape(-1),
        "test_raw": test_raw.reshape(-1),
        "train_scaled": train_scaled,
        "val_scaled": val_scaled,
        "test_scaled": test_scaled,
        "x_tr": x_tr,
        "y_tr": y_tr,
        "x_vl": x_vl,
        "y_vl": y_vl,
        "x_ts": x_ts,
        "y_ts": y_ts,
        "history_before_test_scaled": history_before_test_scaled,
        "history_before_test_raw": np.concatenate([train_raw.reshape(-1), val_raw.reshape(-1)]),
        "actual_future": test_raw.reshape(-1)[:available_horizon],
        "available_horizon": available_horizon,
    }


def inverse_transform_vector(scaler: MinMaxScaler, values_1d):
    values_2d = np.array(values_1d, dtype=float).reshape(-1, 1)
    return scaler.inverse_transform(values_2d).reshape(-1)


def compute_summary_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float).reshape(-1)
    y_pred = np.array(y_pred, dtype=float).reshape(-1)

    rmse = float(np.sqrt(np.mean(np.square(y_true - y_pred))))
    mae = float(np.mean(np.abs(y_true - y_pred)))

    denominator = np.where(np.abs(y_true) < 1e-8, np.nan, np.abs(y_true))
    mape = float(np.nanmean(np.abs((y_true - y_pred) / denominator)) * 100.0)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
    }


def compute_horizon_rows(model_name, run_id, iteration_id, variable_name, y_true, y_pred):
    rows = []
    y_true = np.array(y_true, dtype=float).reshape(-1)
    y_pred = np.array(y_pred, dtype=float).reshape(-1)

    for horizon_idx, (true_value, pred_value) in enumerate(zip(y_true, y_pred), start=1):
        point_metrics = compute_summary_metrics([true_value], [pred_value])
        rows.append({
            "model": model_name,
            "run": run_id,
            "iteration": iteration_id,
            "variable": variable_name,
            "horizon": horizon_idx,
            "RMSE": point_metrics["RMSE"],
            "MAE": point_metrics["MAE"],
            "MAPE": point_metrics["MAPE"],
        })
    return rows


def build_bigru_model(input_length: int, config: dict):
    model = Sequential([
        Bidirectional(GRU(config["gru_units_1"], input_shape=(input_length, 1), return_sequences=True)),
        Bidirectional(GRU(config["gru_units_2"], return_sequences=False)),
        Dense(1, activation="linear"),
    ])
    model.compile(
        optimizer=RMSprop(config["learning_rate"]),
        loss=root_mean_squared_error,
        metrics=[MeanAbsoluteError(), MeanAbsolutePercentageError()],
    )
    return model


def build_lstm_model(input_length: int, config: dict):
    model = Sequential([
        LSTM(config["lstm_units_1"], input_shape=(input_length, 1), return_sequences=True),
        Dropout(config["dropout"]),
        LSTM(config["lstm_units_2"], return_sequences=True),
        LSTM(config["lstm_units_3"], return_sequences=False),
        Dense(config["dense_units"], activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(
        optimizer=RMSprop(config["learning_rate"]),
        loss=root_mean_squared_error,
        metrics=[MeanAbsoluteError(), MeanAbsolutePercentageError()],
    )
    return model


def recursive_forecast_neural(model, scaler, history_scaled, horizon: int, input_length: int):
    if len(history_scaled) < input_length:
        raise ValueError("No hay suficientes observaciones historicas para iniciar el forecast recursivo.")

    window = np.array(history_scaled[-input_length:], dtype=float).reshape(1, input_length, 1)
    predictions_scaled = []

    for _ in range(horizon):
        next_scaled = float(model.predict(window, verbose=0).reshape(-1)[0])
        predictions_scaled.append(next_scaled)
        if input_length == 1:
            window = np.array(next_scaled, dtype=float).reshape(1, 1, 1)
        else:
            window = np.concatenate([window[:, 1:, :], np.array(next_scaled, dtype=float).reshape(1, 1, 1)], axis=1)

    return inverse_transform_vector(scaler, predictions_scaled)


def recursive_forecast_arima(train_scaled, val_scaled, scaler, horizon: int, order, seasonal_order):
    model = ARIMA(order=order, seasonal_order=seasonal_order)

    fit_start = time()
    model.fit(y=pd.Series(train_scaled))
    training_time = time() - fit_start

    if len(val_scaled) > 0:
        model.update(val_scaled)

    predictions_scaled = model.predict(n_periods=horizon)
    predictions = inverse_transform_vector(scaler, np.array(predictions_scaled, dtype=float))

    return predictions, training_time


def make_prepared_folds(folds, target_columns, input_length: int, output_length: int, forecast_horizon: int):
    prepared = {}
    for iteration_id, fold_data in folds.items():
        prepared[iteration_id] = {}
        for col in target_columns:
            prepared[iteration_id][col] = prepare_univariate_fold_data(
                fold_data["train"][col],
                fold_data["val"][col],
                fold_data["test"][col],
                input_length=input_length,
                output_length=output_length,
                forecast_horizon=forecast_horizon,
            )
    return prepared


def export_metrics_to_excel(results_by_model, excel_path: Path, config_dict: dict):
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        pd.DataFrame([config_dict]).to_excel(writer, sheet_name="configuration", index=False)

        for model_name, model_results in results_by_model.items():
            summary_df = pd.DataFrame(model_results["summary"])
            horizon_df = pd.DataFrame(model_results["horizon"])

            if not summary_df.empty:
                summary_df = summary_df.sort_values(["run", "iteration", "variable"]).reset_index(drop=True)
            if not horizon_df.empty:
                horizon_df = horizon_df.sort_values(["run", "iteration", "variable", "horizon"]).reset_index(drop=True)

            summary_df.to_excel(writer, sheet_name=f"{model_name}_summary", index=False)
            horizon_df.to_excel(writer, sheet_name=f"{model_name}_horizon", index=False)


def plot_example_forecast(actual_values, predicted_values, title: str):
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(actual_values) + 1), actual_values, label="Real", linewidth=2.5)
    plt.plot(range(1, len(predicted_values) + 1), predicted_values, label="Forecast", linewidth=2.5)
    plt.xlabel("Horizon")
    plt.ylabel("Value")
    plt.title(title)
    plt.grid(False)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 4. Carga de datos

Se mantiene el mismo `data.csv` del proyecto original y se respeta el recorte temporal utilizado en el notebook base.


In [ ]:
new_order_columns = [
    "Free Memory",
    "Used Memory",
    "Free Disk",
    "Used Disk",
    "Disk read/s",
    "Disk write/s",
    "NetBytes In",
    "NetBytes Out",
    "NetPackets In",
    "NetPackets Out",
    "Rx packets",
    "Tx packets",
    "CPU percent",
    "Memory Used percent",
    "Disk Used percent",
    "Uptime",
]

df = pd.read_csv(DATA_FILE, index_col=0, parse_dates=["beginTimeSeconds"])
df = df[new_order_columns].copy()
df = df[df.index <= CUTOFF_DATE].copy()

if TARGET_COLUMNS is None:
    target_columns = list(df.columns)
else:
    target_columns = [col for col in TARGET_COLUMNS if col in df.columns]

print(df.shape)
print(df.index.min(), df.index.max())
print(target_columns)
df.head()

## 5. Folds temporales

Se conserva el esquema expandible del notebook original: cada iteracion usa un bloque historico creciente y el siguiente bloque temporal como test.


In [ ]:
folds, cut_dates = build_temporal_folds(
    df=df,
    num_folds=NUM_FOLDS,
    train_ratio=TRAIN_WITHIN_FOLD_RATIO,
)

fold_rows = []
for iteration_id, fold_data in folds.items():
    fold_rows.append({
        "iteration": iteration_id,
        "train_start": fold_data["train"].index.min(),
        "train_end": fold_data["train"].index.max(),
        "val_start": fold_data["val"].index.min(),
        "val_end": fold_data["val"].index.max(),
        "test_start": fold_data["test"].index.min(),
        "test_end": fold_data["test"].index.max(),
        "train_rows": len(fold_data["train"]),
        "val_rows": len(fold_data["val"]),
        "test_rows": len(fold_data["test"]),
    })

fold_info_df = pd.DataFrame(fold_rows)
fold_info_df

## 6. Preparacion univariada por fold

Aunque el dataset contiene varias metricas, el entrenamiento sigue siendo **univariado por variable**.

Para cada fold y variable se generan:

- escala `MinMaxScaler(-1, 1)` ajustada solo con train
- datasets supervisados one-step
- historia previa al test para inicializar el forecast recursivo


In [ ]:
prepared_folds = make_prepared_folds(
    folds=folds,
    target_columns=target_columns,
    input_length=INPUT_LENGTH,
    output_length=OUTPUT_LENGTH,
    forecast_horizon=FORECAST_HORIZON,
)

preview_rows = []
for iteration_id, variables_data in prepared_folds.items():
    sample_col = target_columns[0]
    sample = variables_data[sample_col]
    preview_rows.append({
        "iteration": iteration_id,
        "variable": sample_col,
        "x_tr_shape": sample["x_tr"].shape,
        "y_tr_shape": sample["y_tr"].shape,
        "x_vl_shape": sample["x_vl"].shape,
        "y_vl_shape": sample["y_vl"].shape,
        "available_horizon": sample["available_horizon"],
    })

pd.DataFrame(preview_rows)

## 7. Experimentos BiGRU y LSTM

Cada modelo se entrena como **one-step** y luego se usa en inferencia **multi-step recursiva**.

En cada fila resumen se almacenan:

- `run`
- `iteration`
- `variable`
- `RMSE`
- `MAE`
- `MAPE`
- `training_time_seconds`

Ademas se guarda una fila por horizonte en hojas separadas del Excel.


In [ ]:
def run_neural_experiments(model_name, builder_fn, builder_config, prepared_folds, target_columns, runs, base_seed, model_seed_offset, cut_dates):
    summary_records = []
    horizon_records = []
    example_forecasts = {}

    for run_id in range(1, runs + 1):
        print(f"===== {model_name} | run {run_id}/{runs} =====")

        for iteration_id, variables_data in prepared_folds.items():
            print(f"--- Iteration {iteration_id} ---")

            for variable_idx, variable_name in enumerate(target_columns):
                prepared = variables_data[variable_name]
                current_seed = base_seed + run_id * 1000 + iteration_id * 100 + variable_idx * 10 + model_seed_offset
                set_seed(current_seed)
                tf.keras.backend.clear_session()

                model = builder_fn(INPUT_LENGTH, builder_config)

                x_tr = prepared["x_tr"].reshape(-1, INPUT_LENGTH, 1)
                y_tr = prepared["y_tr"].reshape(-1, OUTPUT_LENGTH)
                x_vl = prepared["x_vl"].reshape(-1, INPUT_LENGTH, 1)
                y_vl = prepared["y_vl"].reshape(-1, OUTPUT_LENGTH)

                callbacks = [
                    tf.keras.callbacks.EarlyStopping(
                        monitor="val_loss",
                        patience=builder_config["patience"],
                        restore_best_weights=True,
                    )
                ]

                start_time = time()
                history = model.fit(
                    x=x_tr,
                    y=y_tr,
                    validation_data=(x_vl, y_vl),
                    epochs=builder_config["epochs"],
                    batch_size=builder_config["batch_size"],
                    verbose=0,
                    callbacks=callbacks,
                )
                training_time = time() - start_time

                available_horizon = prepared["available_horizon"]
                predicted = recursive_forecast_neural(
                    model=model,
                    scaler=prepared["scaler"],
                    history_scaled=prepared["history_before_test_scaled"],
                    horizon=available_horizon,
                    input_length=INPUT_LENGTH,
                )
                actual = prepared["actual_future"][:available_horizon]

                metrics = compute_summary_metrics(actual, predicted)
                summary_records.append({
                    "model": model_name,
                    "run": run_id,
                    "iteration": iteration_id,
                    "variable": variable_name,
                    "forecast_horizon": available_horizon,
                    "train_end": cut_dates[iteration_id]["train_end"],
                    "val_end": cut_dates[iteration_id]["val_end"],
                    "test_end": cut_dates[iteration_id]["test_end"],
                    "RMSE": metrics["RMSE"],
                    "MAE": metrics["MAE"],
                    "MAPE": metrics["MAPE"],
                    "training_time_seconds": training_time,
                    "epochs_trained": len(history.history["loss"]),
                    "seed": current_seed,
                })

                horizon_records.extend(
                    compute_horizon_rows(
                        model_name=model_name,
                        run_id=run_id,
                        iteration_id=iteration_id,
                        variable_name=variable_name,
                        y_true=actual,
                        y_pred=predicted,
                    )
                )

                example_key = (model_name, iteration_id, variable_name)
                if example_key not in example_forecasts:
                    example_forecasts[example_key] = {
                        "actual": actual,
                        "predicted": predicted,
                    }

    return {
        "summary": summary_records,
        "horizon": horizon_records,
        "examples": example_forecasts,
    }


bigru_results = run_neural_experiments(
    model_name="BiGRU",
    builder_fn=build_bigru_model,
    builder_config=BIGRU_CONFIG,
    prepared_folds=prepared_folds,
    target_columns=target_columns,
    runs=RUNS,
    base_seed=BASE_SEED,
    model_seed_offset=1,
    cut_dates=cut_dates,
)

lstm_results = run_neural_experiments(
    model_name="LSTM",
    builder_fn=build_lstm_model,
    builder_config=LSTM_CONFIG,
    prepared_folds=prepared_folds,
    target_columns=target_columns,
    runs=RUNS,
    base_seed=BASE_SEED,
    model_seed_offset=2,
    cut_dates=cut_dates,
)

print("BiGRU summary rows:", len(bigru_results["summary"]))
print("LSTM summary rows:", len(lstm_results["summary"]))

## 8. Experimentos ARIMA

Para ARIMA se mantiene el enfoque univariado. Se ajusta sobre `train` y se actualiza con `val` antes de lanzar el forecast recursivo multi-step hacia `test`.


In [ ]:
def run_arima_experiments(prepared_folds, target_columns, runs, cut_dates):
    summary_records = []
    horizon_records = []
    example_forecasts = {}

    for run_id in range(1, runs + 1):
        print(f"===== ARIMA | run {run_id}/{runs} =====")

        for iteration_id, variables_data in prepared_folds.items():
            print(f"--- Iteration {iteration_id} ---")

            for variable_name in target_columns:
                prepared = variables_data[variable_name]
                available_horizon = prepared["available_horizon"]

                predicted, training_time = recursive_forecast_arima(
                    train_scaled=prepared["train_scaled"],
                    val_scaled=prepared["val_scaled"],
                    scaler=prepared["scaler"],
                    horizon=available_horizon,
                    order=ARIMA_ORDER,
                    seasonal_order=ARIMA_SEASONAL_ORDER,
                )

                actual = prepared["actual_future"][:available_horizon]
                metrics = compute_summary_metrics(actual, predicted)

                summary_records.append({
                    "model": "ARIMA",
                    "run": run_id,
                    "iteration": iteration_id,
                    "variable": variable_name,
                    "forecast_horizon": available_horizon,
                    "train_end": cut_dates[iteration_id]["train_end"],
                    "val_end": cut_dates[iteration_id]["val_end"],
                    "test_end": cut_dates[iteration_id]["test_end"],
                    "RMSE": metrics["RMSE"],
                    "MAE": metrics["MAE"],
                    "MAPE": metrics["MAPE"],
                    "training_time_seconds": training_time,
                    "seed": np.nan,
                })

                horizon_records.extend(
                    compute_horizon_rows(
                        model_name="ARIMA",
                        run_id=run_id,
                        iteration_id=iteration_id,
                        variable_name=variable_name,
                        y_true=actual,
                        y_pred=predicted,
                    )
                )

                example_key = ("ARIMA", iteration_id, variable_name)
                if example_key not in example_forecasts:
                    example_forecasts[example_key] = {
                        "actual": actual,
                        "predicted": predicted,
                    }

    return {
        "summary": summary_records,
        "horizon": horizon_records,
        "examples": example_forecasts,
    }


arima_results = run_arima_experiments(
    prepared_folds=prepared_folds,
    target_columns=target_columns,
    runs=RUNS,
    cut_dates=cut_dates,
)

print("ARIMA summary rows:", len(arima_results["summary"]))

## 9. Exporte a Excel

Se genera un unico archivo Excel con hojas resumen y por horizonte para cada modelo.


In [ ]:
results_by_model = {
    "BiGRU": {
        "summary": bigru_results["summary"],
        "horizon": bigru_results["horizon"],
    },
    "LSTM": {
        "summary": lstm_results["summary"],
        "horizon": lstm_results["horizon"],
    },
    "ARIMA": {
        "summary": arima_results["summary"],
        "horizon": arima_results["horizon"],
    },
}

config_export = {
    "data_file": str(DATA_FILE),
    "runs": RUNS,
    "num_folds": NUM_FOLDS,
    "train_within_fold_ratio": TRAIN_WITHIN_FOLD_RATIO,
    "input_length": INPUT_LENGTH,
    "output_length": OUTPUT_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "cutoff_date": CUTOFF_DATE,
    "target_columns": ", ".join(target_columns),
    "bigru_config": str(BIGRU_CONFIG),
    "lstm_config": str(LSTM_CONFIG),
    "arima_order": str(ARIMA_ORDER),
    "arima_seasonal_order": str(ARIMA_SEASONAL_ORDER),
}

export_metrics_to_excel(
    results_by_model=results_by_model,
    excel_path=EXCEL_PATH,
    config_dict=config_export,
)

print(f"Excel generado en: {EXCEL_PATH.resolve()}")

## 10. Vista rapida de resultados

Las siguientes celdas permiten revisar las primeras filas de las hojas resumen y, opcionalmente, visualizar un ejemplo de forecast recursivo por modelo.


In [ ]:
bigru_summary_df = pd.DataFrame(bigru_results["summary"])
lstm_summary_df = pd.DataFrame(lstm_results["summary"])
arima_summary_df = pd.DataFrame(arima_results["summary"])

print("BiGRU")
display(bigru_summary_df.head())
print("LSTM")
display(lstm_summary_df.head())
print("ARIMA")
display(arima_summary_df.head())

In [ ]:
example_variable = target_columns[0]
example_iteration = 0

plot_example_forecast(
    actual_values=bigru_results["examples"][("BiGRU", example_iteration, example_variable)]["actual"],
    predicted_values=bigru_results["examples"][("BiGRU", example_iteration, example_variable)]["predicted"],
    title=f"BiGRU recursive multi-step | iteration {example_iteration} | {example_variable}",
)

plot_example_forecast(
    actual_values=lstm_results["examples"][("LSTM", example_iteration, example_variable)]["actual"],
    predicted_values=lstm_results["examples"][("LSTM", example_iteration, example_variable)]["predicted"],
    title=f"LSTM recursive multi-step | iteration {example_iteration} | {example_variable}",
)

plot_example_forecast(
    actual_values=arima_results["examples"][("ARIMA", example_iteration, example_variable)]["actual"],
    predicted_values=arima_results["examples"][("ARIMA", example_iteration, example_variable)]["predicted"],
    title=f"ARIMA recursive multi-step | iteration {example_iteration} | {example_variable}",
)